In [2]:
import pandas as pd

------

In [17]:
ATC_mapping = {
    "A": "Alimentary tract and metabolism",
    "B": "Blood and blood-forming organs",
    "C": "Cardiovascular system",
    "D": "Dermatologicals",
    "G": "Genito-urinary system and sex hormones",
    "H": "Systemic hormonal preparations, excluding sex hormones and insulins",
    "J": "Antiinfectives for systemic use",
    "L": "Antineoplastic and immunomodulating agents",
    "M": "Musculoskeletal system",
    "N": "Nervous system",
    "P": "Antiparasitic products, insecticides and repellents",
    "R": "Respiratory system",
    "S": "Sensory organs",
    "V": "Various"}

def apply_atc_mapping(atc_code):
    """Map ATC code to its description."""
    if atc_code in ATC_mapping:
        return ATC_mapping[atc_code]
    else:
        return "Unknown"

In [3]:
def txt_into_df(txt_file, columns):
    data = []
    with open(txt_file, "r", encoding="utf-8") as file:
        file_list = file.readlines()
        for line in file_list:
            line = line.strip().split("\",\"")
            line = [item.replace('"', '') for item in line]
            data.append(line)

    df = pd.DataFrame(data, columns=columns)
    df = df.drop(columns=["_"])
    return df

In [4]:
drug = "./../data/HealthCanada/allfiles/drug.txt"
ingred = "./../data/HealthCanada/allfiles/ingred.txt"
form = "./../data/HealthCanada/allfiles/form.txt"
route = "./../data/HealthCanada/allfiles/route.txt"
status = "./../data/HealthCanada/allfiles/status.txt"
ther = "./../data/HealthCanada/allfiles/ther.txt"

In [5]:

columns_drug=["DRUG_CODE", "_", "CLASS", "DRUG_ID", "BRAND_NAME", "_", "_", "_", "_", "_", "_", "_", "_", "_"]
df_drug = txt_into_df(drug, columns_drug)
df_drug.head()

,DRUG_CODE,CLASS,DRUG_ID,BRAND_NAME
0,9,Human,00015741,TAPAZOLE
1,15,Human,00015229,AVENTYL
2,16,Human,00015237,AVENTYL
3,57,Human,00050520,MINERALE LEGERE HUILE
4,68,Human,00050466,METHYLENE BLEU LIQ 1%


In [7]:
columns_ingred = ["DRUG_CODE", "_", "INGREDIENT", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_", "_"]
df_ingred = txt_into_df(ingred, columns_ingred)
df_ingred.head()

,DRUG_CODE,INGREDIENT
0,9,METHIMAZOLE
1,15,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE)
2,16,NORTRIPTYLINE (NORTRIPTYLINE HYDROCHLORIDE)
3,57,MINERAL OIL LIGHT
4,68,METHYLENE BLUE


In [8]:
columns_form = ["DRUG_CODE", "_", "PHARMACEUTICAL_FORM", "_"]
df_form = txt_into_df(form, columns_form)
df_form.head()

,DRUG_CODE,PHARMACEUTICAL_FORM
0,9,TABLET
1,15,CAPSULE
2,16,CAPSULE
3,57,LIQUID
4,68,LIQUID


In [9]:
columns_route = ["DRUG_CODE", "_", "ROUTE_OF_ADMINISTRATION", "_"]
df_route = txt_into_df(route, columns_route)
df_route.head()

,DRUG_CODE,ROUTE_OF_ADMINISTRATION
0,9,ORAL
1,15,ORAL
2,16,ORAL
3,57,TOPICAL
4,68,ORAL


In [10]:
columns_status = ["DRUG_CODE", "_", "STATUS", "HISTORY_DATE", "_", "_", "_"]
df_status = txt_into_df(status, columns_status)
df_status.head()

,DRUG_CODE,STATUS,HISTORY_DATE
0,9,MARKETED,31-DEC-1951
1,9,MARKETED,16-APR-2001
2,9,APPROVED,24-JAN-2001
3,9,APPROVED,18-JAN-2001
4,9,APPROVED,07-MAR-2025


In [19]:
columns_ther = ["DRUG_CODE", "TC_ATC_NUMBER", "_", "_"]
df_ther = txt_into_df(ther, columns_ther)
df_ther["ATC_CODE"] = df_ther["TC_ATC_NUMBER"].str[:1]
df_ther["ATC_DESCRIPTION"] = df_ther["ATC_CODE"].apply(apply_atc_mapping)
df_ther.head(10)

,DRUG_CODE,TC_ATC_NUMBER,ATC_CODE,ATC_DESCRIPTION
0,9,H03BB02,H,"Systemic hormonal preparations, excluding sex ..."
1,15,N06AA10,N,Nervous system
2,16,N06AA10,N,Nervous system
3,57,D02AC,D,Dermatologicals
4,68,V04CG05,V,Various
5,69,V03AB17,V,Various
6,115,V07AB,V,Various
7,117,V07AB,V,Various
8,160,A03AD01,A,Alimentary tract and metabolism
9,193,R06AA11,R,Respiratory system


In [22]:
print(df_ther["DRUG_CODE"].nunique())
print(len(df_ther))

12544
13287


In [39]:
d = {}
# create a dict with drug codes as keys and ATC descriptions as values
for index, row in df_ther.iterrows():
    drug_code = row["DRUG_CODE"]
    atc_description = row["ATC_DESCRIPTION"]
    if drug_code not in d.keys():
        d[drug_code] = []
    d[drug_code].append(atc_description)
    

df_drug_from_dict = pd.DataFrame.from_dict(
    {k: "; ".join(v) for k, v in d.items()},
    orient="index",
    columns=["ATC_DESCRIPTIONS"]
).reset_index().rename(columns={"index": "DRUG_CODE"})

df_drug_from_dict.head()

,DRUG_CODE,ATC_DESCRIPTIONS
0,9,"Systemic hormonal preparations, excluding sex ..."
1,15,Nervous system
2,16,Nervous system
3,57,Dermatologicals
4,68,Various


----

In [41]:
import requests

resp = requests.get(
    "https://api.fda.gov/drug/label.json?search=openfda.brand_name:metformin"
)
data = resp.json()
# print(data['results'][0]['indications_and_usage'][0])
print(print(data['results'][0].keys()))


dict_keys(['spl_product_data_elements', 'boxed_warning', 'indications_and_usage', 'dosage_and_administration', 'dosage_forms_and_strengths', 'contraindications', 'warnings_and_cautions', 'adverse_reactions', 'adverse_reactions_table', 'drug_interactions', 'drug_interactions_table', 'use_in_specific_populations', 'pregnancy', 'pediatric_use', 'geriatric_use', 'overdosage', 'description', 'clinical_pharmacology', 'clinical_pharmacology_table', 'mechanism_of_action', 'pharmacokinetics', 'pharmacokinetics_table', 'nonclinical_toxicology', 'carcinogenesis_and_mutagenesis_and_impairment_of_fertility', 'clinical_studies', 'clinical_studies_table', 'how_supplied', 'information_for_patients', 'spl_patient_package_insert', 'package_label_principal_display_panel', 'set_id', 'id', 'effective_time', 'version', 'openfda'])
None
